# Optimizers: momentum, RMSProp and Adam

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The trouble with plain gradient descent
In narrow valleys (when the curvature differs a lot between directions) plain gradient descent **zigzags**: it oscillates along the steep direction and creeps along the shallow one.

> [Four optimizers from the same start on the narrow valley $f=\frac12x^2+12.5y^2$. The red star is the minimum.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_optimizers.png)

*Four optimizers from the same start on the narrow valley $f=\frac12x^2+12.5y^2$. The red star is the minimum.*

## Momentum: a ball that picks up speed
Instead of stepping directly, we update a **velocity** that is a running average of past gradients:

$$v\leftarrow\beta v-\eta\nabla L\qquad\theta\leftarrow\theta+v$$

- Along a direction where gradients keep the same sign, the velocity **builds up**.
- Along a direction where the sign flips (the zigzag), they **cancel**.

With $\beta=0.9$ the final speed is up to $\dfrac{1}{1-\beta}=10$ times a plain step.

## Momentum by numbers
Constant gradient $g=1$, $\eta=0.1$, $\beta=0.9$:

| Step | $v=0.9v-0.1$ | Total movement |
|---|---|---|
| 1 | −0.100 | −0.100 |
| 2 | −0.190 | −0.290 |
| 3 | −0.271 | −0.561 |
| 4 | −0.344 | −0.905 |
| ∞ | −1.000 | |

## RMSProp: a separate learning rate for each parameter
It keeps a running average of the **squared** gradient and divides the step by its square root:

$$s\leftarrow\rho s+(1-\rho)(\nabla L)^2\qquad\theta\leftarrow\theta-\frac{\eta}{\sqrt{s}+\epsilon}\nabla L$$

A parameter with large gradients takes smaller steps and the other way round: automatic scaling.

## Adam = momentum + RMSProp
| Step | Formula | Meaning |
|---|---|---|
| 1 | $m\leftarrow\beta_1m+(1-\beta_1)g$ | Average gradient (direction) |
| 2 | $s\leftarrow\beta_2s+(1-\beta_2)g^2$ | Average squared gradient (scale) |
| 3 | $\hat{m}=\dfrac{m}{1-\beta_1^t},\quad\hat{s}=\dfrac{s}{1-\beta_2^t}$ | Corrects the bias of starting at zero |
| 4 | $\theta\leftarrow\theta-\eta\dfrac{\hat{m}}{\sqrt{\hat{s}}+\epsilon}$ | The step |

Defaults: $\beta_1=0.9$, $\beta_2=0.999$, $\eta=0.001$, $\epsilon=10^{-8}$.

## Worked example: the first Adam step
With $g=4$ and $\eta=0.1$:

| Quantity | Calculation | Value |
|---|---|---|
| $m$ | $0.1\times4$ | 0.4 |
| $s$ | $0.001\times16$ | 0.016 |
| $\hat{m}$ | $0.4/(1-0.9)$ | 4 |
| $\hat{s}$ | $0.016/(1-0.999)$ | 16 |
| Step | $0.1\times4/\sqrt{16}$ | 0.1 |

> A striking result: the first Adam step is always about $\eta$, **whatever the size of the gradient**. With a gradient of 4000 the step would still be 0.1. This is why Adam does not care about the scale of the loss and its default learning rate often just works.

## Comparison
| Optimizer | Extra memory | Strength | Weakness |
|---|---|---|---|
| SGD | None | Simple; generalizes well | Slow in valleys; sensitive to η |
| SGD + momentum | v | Faster; common in computer vision | One more hyperparameter |
| RMSProp | s | Good for recurrent networks | No momentum |
| Adam | m and s | A good default; little tuning | Sometimes generalizes slightly worse |
| AdamW | m and s | Adam with correct weight decay; the standard for transformers | |

## Learning-rate schedules
| Schedule | Behaviour | Used in |
|---|---|---|
| Constant | Always η | Simple cases |
| Step | Divide by 10 every few epochs | Classic convolutional networks |
| Cosine | Smooth decay to zero | Modern training |
| Warmup | Start at zero and ramp up | Transformers |
| Reduce on plateau | When the validation loss stops improving | General |



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f    = lambda p: 0.5 * p[0] ** 2 + 12.5 * p[1] ** 2       # a narrow valley: 25 times steeper along y
grad = lambda p: np.array([p[0], 25 * p[1]])

def run(name, steps=80):
    p = np.array([-3.5, 1.5]); path = [p.copy()]
    v = np.zeros(2); s = np.zeros(2); m = np.zeros(2)
    for t in range(1, steps + 1):
        g = grad(p)
        if name == "GD":
            p = p - 0.07 * g                      # must stay below 2/25 = 0.08 or it diverges
        elif name == "Momentum":
            v = 0.8 * v - 0.02 * g; p = p + v
        elif name == "RMSProp":
            s = 0.9 * s + 0.1 * g ** 2; p = p - 0.1 * g / (np.sqrt(s) + 1e-8)
        elif name == "Adam":
            m = 0.9 * m + 0.1 * g; s = 0.999 * s + 0.001 * g ** 2
            p = p - 0.2 * (m / (1 - 0.9 ** t)) / (np.sqrt(s / (1 - 0.999 ** t)) + 1e-8)
        path.append(p.copy())
    return np.array(path)

u, w = np.meshgrid(np.linspace(-4, 1.5, 200), np.linspace(-2, 2, 200))
plt.contour(u, w, 0.5 * u ** 2 + 12.5 * w ** 2, levels=12, colors="lightgray")
for name in ["GD", "Momentum", "RMSProp", "Adam"]:
    path = run(name)
    plt.plot(path[:, 0], path[:, 1], marker="o", ms=2, label=name)
    values = np.array([f(q) for q in path])
    hit = np.argmax(values < 0.01) if (values < 0.01).any() else "never"
    print(f"{name:9s} f after 80 steps = {values[-1]:.2e}   first step with f < 0.01: {hit}")
plt.legend(); plt.title("Optimizers on a narrow valley"); plt.show()

# Momentum with a constant gradient: the velocity approaches -eta / (1 - beta)
v = 0.0
for step in range(1, 5):
    v = 0.9 * v - 0.1 * 1.0
    print(f"step {step}: v = {v:.3f}")

# The first Adam step has size eta, whatever the gradient
for g in [0.004, 4.0, 4000.0]:
    m = 0.1 * g; s = 0.001 * g ** 2
    print(f"g = {g:8}: first Adam step = {0.1 * (m / 0.1) / np.sqrt(s / 0.001):.4f}")

### Exercise

With $\beta=0.5$, $\eta=0.1$ and a constant gradient $g=2$, compute the momentum velocity for three steps. What is the final speed?

In [ ]:
# Your answer here

### Exercise

In the gradient-descent lab (earlier lesson) run the narrow valley with $\eta=0.15$ without momentum, then with $\beta=0.8$.

In [ ]:
# Your answer here